## 🔎 Introduction

This notebook provides a clear, practical comparison between `class methods`, `scopes`, and `enums` in Ruby and Rails.
Using a simple Task model, we explore how each approach handles filtering and state management, showing when and why you might choose one over the other.

### Our approach is to reproduced a minimal Rails-like environment:

 - `Model` logic

 - `Controller` instance variables

-  `ERB` rendering

 - Separation of concerns

 - Teaching how Rails works internally

This makes your notebook an excellent educational tool.

## Summary
### ✔️ Class Methods
- Pure Ruby
- Allow complex logic
- Accept parameters

### ✔️ Scopes (Rails)
- Chainable
- Short, expressive
- Ideal for simple filters

### ✔️ Enums
- Map states to integers
- Auto‑generate methods: `task.completed?`, `.completed`
- Useful for finite status values


**Tutorial**: [Scopes vs Enums in Rails](https://medium.com/jungletronics/scopes-vs-enums-in-rails-60ee5a7c6356) - Which One Should You Use?

### 🧱 Step-by-Step: What We Did

#### ① Create a plain Ruby version of a Task model

We implemented a simplified Task class:

 - memory storage (`@@tasks`)

 - attributes (`title` and `status`)

- class methods resembling Rails scopes (`completed`, `not_completed`)

This lets us simulate how `scopes/class` methods behave without `ActiveRecord`.

✔ Good for teaching the concepts of scopes vs class methods.

## Class Methods Example
In Rails models, class methods allow custom logic with parameters.

In [1]:
class Task
  attr_accessor :title, :status

  @@tasks = []

  def initialize(title:, status:)
    @title = title
    @status = status
    @@tasks << self
  end

  def self.all
    @@tasks
  end

  def self.completed
    all.select { |t| t.status == :completed }
  end

  def self.not_completed
    all.select { |t| t.status == :not_completed }
  end
end

:not_completed

#### ② Seed two Task objects

This simulates Task.create(...) in Rails.

In [2]:
Task.new(title: "Clean Car",  status: :completed)
Task.new(title: "Study Ruby", status: :not_completed)


#<#<Class:0x00007767261da830>::Task:0x00007767289e6ae0 @title="Study Ruby", @status=:not_completed>

#### ③ Create a fake TasksController

Here’s the key point:

➤ Rails views always depend on instance variables assigned in controllers.

A typical Rails controller action:

In [3]:
# Stub for Rails controller base class (needed outside Rails)
class ApplicationController
end

class TasksController < ApplicationController
  def index
    @completed_tasks     = Task.completed
    @not_completed_tasks = Task.not_completed
  end
end


:index

#### 🎯 So Why Did We Need a Controller?

Because:

 - The ERB template expects instance variables (`@completed_tasks`)

 - Instance variables must come from a controller action, just like in Rails

 - We needed the same architecture Rails uses:

     `Controller` → sets the variables

      `View` → consumes them

Otherwise our ERB view couldn’t work at all.

The controller puts data into `@completed_tasks`, and Rails injects these into the view’s binding.

But since Rails is NOT running, we had to:

 - create a stub for `ApplicationController`

 - build our own `TasksController`

 - define an `index` action

 - call `controller.index` manually

This is the only way to produce the instance variables the view depends on.

Without these instance variables, our ERB view would raise:

#### ④ Manually render an ERB template

In Rails:

Rails automatically finds `app/views/tasks/index.html.erb`

Rails automatically __binds instance variables__ into the `view`

In our notebook, __we must do this ourselves__:
```ruby
renderer.result(controller.instance_eval { binding })
```

__This is critical__:

`renderer.result` requires a binding (variable context)

Only the `controller` instance contains `@completed_tasks` and `@not_completed_tasks`

So we inject the `controller’s` binding into the `ERB` template

→ exactly what Rails does behind the scenes

In [4]:
index_view = <<~ERB
  <h1 class="mb-4">Listing Todos</h1>

  <div class="row">
    <div class="col-md-6">
      <h3 class="text-success">Completed Tasks</h3>
      <ul class="list-group mb-4">
        <% @completed_tasks.each do |task| %>
          <li class="list-group-item"><%= task.title %></li>
        <% end %>
      </ul>
    </div>

    <div class="col-md-6">
      <h3 class="text-danger">Not Completed Tasks</h3>
      <ul class="list-group mb-4">
        <% @not_completed_tasks.each do |task| %>
          <li class="list-group-item"><%= task.title %></li>
        <% end %>
      </ul>
    </div>
  </div>
ERB

"<h1 class=\"mb-4\">Listing Todos</h1>\n\n<div class=\"row\">\n  <div class=\"col-md-6\">\n    <h3 class=\"text-success\">Completed Tasks</h3>\n    <ul class=\"list-group mb-4\">\n      <% @completed_tasks.each do |task| %>\n        <li class=\"list-group-item\"><%= task.title %></li>\n      <% end %>\n    </ul>\n  </div>\n\n  <div class=\"col-md-6\">\n    <h3 class=\"text-danger\">Not Completed Tasks</h3>\n    <ul class=\"list-group mb-4\">\n      <% @not_completed_tasks.each do |task| %>\n        <li class=\"list-group-item\"><%= task.title %></li>\n      <% end %>\n    </ul>\n  </div>\n</div>\n"

In [5]:
require "erb"

controller = TasksController.new
controller.index

renderer = ERB.new(index_view)
html = renderer.result(controller.instance_eval { binding })  



"<h1 class=\"mb-4\">Listing Todos</h1>\n\n<div class=\"row\">\n  <div class=\"col-md-6\">\n    <h3 class=\"text-success\">Completed Tasks</h3>\n    <ul class=\"list-group mb-4\">\n      \n        <li class=\"list-group-item\">Clean Car</li>\n      \n    </ul>\n  </div>\n\n  <div class=\"col-md-6\">\n    <h3 class=\"text-danger\">Not Completed Tasks</h3>\n    <ul class=\"list-group mb-4\">\n      \n        <li class=\"list-group-item\">Study Ruby</li>\n      \n    </ul>\n  </div>\n</div>\n"

In [6]:
File.write("tasks.html", html)


451

#### This will open HTML in your system browser:

In [7]:
system("xdg-open tasks.html")


true

## Scope Example
Scopes are chainable and perfect for simple queries in Rails.

In [8]:
class Task
  attr_accessor :title, :status

  @@tasks = []

  def initialize(title:, status:)
    @title  = title
    @status = status
    @@tasks << self
  end

  def self.all
    @@tasks
  end

  # --- Scope-like class methods ---
  def self.completed
    all.select { |t| t.status == :completed }
  end

  def self.not_completed
    all.select { |t| t.status == :not_completed }
  end

  # --- Search method ---
  def self.search_by_status(keyword)
    case keyword.to_s
    when "completed"
      completed
    when "not_completed"
      not_completed
    else
      []
    end
  end
end

# Seed data
Task.new(title: "Wash car", status: :completed)
Task.new(title: "Study Ruby", status: :not_completed)

puts "All tasks: #{Task.all.map(&:title)}"
puts "Completed: #{Task.search_by_status('completed').map(&:title)}"
puts "Not Completed: #{Task.search_by_status('not_completed').map(&:title)}"


All tasks: ["Wash car", "Study Ruby"]
Completed: ["Wash car"]
Not Completed: ["Study Ruby"]


#### ✅ Scope Example (Plain Ruby Simulation)

In Rails, scopes are small query helpers used to filter records.
Because we're outside Rails, we simulate scopes using class methods.

What this example does:

 - Stores all tasks in memory using @@tasks

 - Defines two scope-like class methods:

        `completed` → returns tasks with status :completed

    `not_completed` → returns tasks with status :not_completed

Adds a `search_by_status` class method that selects the correct “scope” based on a keyword

Why this mirrors Rails scopes:

In Rails, we would write:
```ruby
scope :completed, -> { where(status: :completed) }
scope :not_completed, -> { where(status: :not_completed) }
````

Since we’re in pure Ruby, our version uses select instead of SQL.

What the output demonstrates
```ruby
Task.search_by_status("completed")
# => ["Wash car"]

Task.search_by_status("not_completed")
# => ["Study Ruby"]

```
It shows how scopes allow:

Clean, readable query shortcuts

A single search method to route to the right scope

No repetition of filtering logic

## Enum Example
Enums map symbolic states to integers and generate helper methods automatically.

In [9]:
# Simulate a minimal Rails-like model without ActiveRecord
class Task
  attr_accessor :title, :done, :status

  # Fake "database"
  @@records = []

  def initialize(title:, done:, status:)
    @title  = title
    @done   = done
    @status = status
  end

  def self.all
    @@records
  end

  def self.seed
    @@records = [
      Task.new(title: "Study Ruby",  done: false, status: :not_completed),
      Task.new(title: "Clear Car",   done: true,  status: :completed),
    ]
  end
end

# Seed the "database"
Task.seed

puts "Seeded tasks:"
Task.all.each { |t| puts "- #{t.title}: #{t.status}" }

Seeded tasks:
- Study Ruby: not_completed
- Clear Car: completed


[#<#<Class:0x00007767261da830>::Task:0x00007767289c9508 @title="Study Ruby", @done=false, @status=:not_completed>, #<#<Class:0x00007767261da830>::Task:0x00007767289c9468 @title="Clear Car", @done=true, @status=:completed>]

In [10]:
module Enum
  def enum(name, values)
    mapping = values.each_with_index.to_h

    # reader, e.g. task.status → :completed
    define_method(name) { @status }

    values.each do |value|
      # predicate: task.completed?
      define_method("#{value}?") { @status == value }

      # bang version: task.completed!
      define_method("#{value}!") do
        @status = value
      end

      # class scope: Task.completed → [tasks]
      define_singleton_method(value) do
        all.select { |t| t.status == value }
      end
    end
  end
end

class Task
  extend Enum
  enum :status, [:not_completed, :completed]
end

task = Task.all.first

puts "Before: #{task.status}"
task.completed!
puts "After completed!: #{task.status}"

puts "Task.completed => #{Task.completed.map(&:title)}"
puts "Task.not_completed => #{Task.not_completed.map(&:title)}"

Before: not_completed
After completed!: completed
Task.completed => ["Study Ruby", "Clear Car"]
Task.not_completed => []


#### ✅ Enum Example (Simulating Rails Enums in Plain Ruby)

Rails enums turn a _symbol-based_ field (like `:completed`) into:

 - predicate methods (completed?)

 - bang setter methods (completed!)

 - automatic scopes (Task.completed)

 - a consistent mapping between symbols and stored values

Since we're not using `ActiveRecord`, WE manually re-created these `enum` features.

`CELL 1` — Minimal “model” with a status attribute

We define a simple Task class with:
```
 - title

 - done (unused but realistic)

 - status (the field that will behave like an enum)

 - a fake database in @@records
```
Then we seed two tasks.

This sets up the data that the enum system will operate on.

`CELL 2` — Implementing our own Enum system

We create a module Enum that mimics Rails’ `enum` feature.

When we call:
```ruby
enum :status, [:not_completed, :completed]
```

Our module dynamically creates four groups of features:

#### ① Reader method
```ruby
task.status
```

Returns the current enum value (symbol).

#### ② Predicate methods

For each `enum`:
```ruby
task.completed?      # true/false
task.not_completed?  # true/false
```

Just like Rails does.

#### ③ Bang setter methods
task.completed!      # sets status to :completed


This simulates:
```ruby
task.status = :completed
```

but in a more Rails-like way.

#### ④ Class-level scopes

We autogenerate methods:
```ruby
Task.completed
Task.not_completed
```

which filter tasks based on the enum’s current value.
Again, exactly like Rails enums.

The demonstration: 
```ruby
puts `"Before: #{task.status}"`
task.completed!
puts "After completed!: #{task.status}"
```

Shows the bang setter in action.
```ruby
Task.completed.map(&:title)
Task.not_completed.map(&:title)
```

Shows the auto-generated enum scopes working.

🎯 In short

Our enum simulation replicates the most important features of Rails enums, but implemented in plain Ruby:


| Feature                     | Rails enum | Your Enum module |
| --------------------------- | ---------- | ---------------- |
| Symbol → method mapping     | ✔️         | ✔️               |
| `status?` predicate methods | ✔️         | ✔️               |
| `status!` setters           | ✔️         | ✔️               |
| Auto-generated scopes       | ✔️         | ✔️               |
| Integer mapping             | ✔️         | ❌ (not needed)   |


### 🙏 Thank You for Reading!

Thanks for taking the time to explore these examples and dive into how class methods, scopes, and enums work in Ruby and Rails.
I hope this notebook helped clarify the concepts and made the inner mechanics easier to understand.

If you found it useful, feel free to build on it, experiment, and make it your own.
Happy coding! 🚀